In [3]:
import torch
from diffusers import StableVideoDiffusionPipeline, StableDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image
import numpy as np

# Low VRAM configuration (6GB VRAM) - Using Stable Video Diffusion
device = "cuda" if torch.cuda.is_available() else "cpu"

# Clear CUDA cache
if device == "cuda":
    torch.cuda.empty_cache()

# Load SVD pipeline with low memory settings
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe = pipe.to(device)
pipe.enable_model_cpu_offload()

# Camera control function for prompts
def build_camera_prompt(scene_description, camera_motion, camera_angle="medium shot", camera_movement="smooth"):
    camera_terms = {
        "zoom_in": "camera slowly zooms in, close-up",
        "zoom_out": "camera slowly zooms out, wide shot",
        "pan_left": "camera pans left, smooth horizontal movement",
        "pan_right": "camera pans right, smooth horizontal movement",
        "tilt_up": "camera tilts up, upward angle",
        "tilt_down": "camera tilts down, downward angle",
        "rotate": "camera rotates around subject, orbital movement",
        "dolly_in": "camera moves forward, dolly in",
        "dolly_out": "camera moves backward, dolly out"
    }
    motion_desc = camera_terms.get(camera_motion, camera_motion)
    return f"{scene_description}. {motion_desc}, {camera_angle}, {camera_movement} camera movement."

# Step 1: Generate initial image from text with camera control
scene = "A cat sitting on a wooden boat in a calm lake"
camera_action = "zoom_in"
angle = "low angle"
movement = "smooth"

prompt = build_camera_prompt(scene, camera_action, angle, movement)

# Load lightweight text-to-image model for initial frame
if device == "cuda":
    torch.cuda.empty_cache()

text_to_image = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    variant="fp16"
)
text_to_image = text_to_image.to(device)
text_to_image.enable_model_cpu_offload()

# Generate initial image
image = text_to_image(
    prompt=prompt,
    num_inference_steps=20,
    height=576,
    width=1024,
    generator=torch.Generator(device).manual_seed(42),
).images[0]

# Clear memory
del text_to_image
if device == "cuda":
    torch.cuda.empty_cache()

# Step 2: Generate video from image using SVD
# Pass PIL Image directly (SVD accepts PIL Image)
frames = pipe(
    image,
    decode_chunk_size=2,
    num_frames=14,
    num_inference_steps=25,
    motion_bucket_id=127,
    generator=torch.Generator(device).manual_seed(42),
).frames[0]

export_to_video(frames, "output2.mp4", fps=7)
print(f"Video saved! Camera motion: {camera_action}, Total frames: {len(frames)}")


100%|██████████| 25/25 [09:55<00:00, 23.83s/it]
It is recommended to use `export_to_video` with `imageio` and `imageio-ffmpeg` as a backend. 
These libraries are not present in your environment. Attempting to use legacy OpenCV backend to export video. 
Support for the OpenCV backend will be deprecated in a future Diffusers version


Video saved! Camera motion: zoom_in, Total frames: 14


In [2]:
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_video

# Low VRAM configuration
device = "cuda" if torch.cuda.is_available() else "cpu"

# Clear cache
if device == "cuda":
    torch.cuda.empty_cache()

# Load AnimateDiff (much faster and lighter than SVD)
motion_adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-2",
    torch_dtype=torch.float16
)

pipe = AnimateDiffPipeline.from_pretrained(
    "frankjoshua/toonyou_beta6",
    motion_adapter=motion_adapter,
    torch_dtype=torch.float16,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)
pipe.enable_model_cpu_offload()  # Critical for 6GB VRAM

# Camera control function
def build_camera_prompt(scene_description, camera_motion, camera_angle="medium shot", camera_movement="smooth"):
    camera_terms = {
        "zoom_in": "camera slowly zooms in, close-up",
        "zoom_out": "camera slowly zooms out, wide shot",
        "pan_left": "camera pans left, smooth horizontal movement",
        "pan_right": "camera pans right, smooth horizontal movement",
        "tilt_up": "camera tilts up, upward angle",
        "tilt_down": "camera tilts down, downward angle",
        "rotate": "camera rotates around subject, orbital movement",
        "dolly_in": "camera moves forward, dolly in",
        "dolly_out": "camera moves backward, dolly out"
    }
    motion_desc = camera_terms.get(camera_motion, camera_motion)
    return f"{scene_description}. {motion_desc}, {camera_angle}, {camera_movement} camera movement."

# Generate video with camera control
scene = "A cat sitting on a wooden boat in a calm lake"
camera_action = "zoom_in"
angle = "low angle"
movement = "smooth"

prompt = build_camera_prompt(scene, camera_action, angle, movement)

# Generate video (much faster than SVD!)
frames = pipe(
    prompt=prompt,
    num_inference_steps=20,  # Can go lower for speed
    guidance_scale=7.5,
    num_frames=16,  # Adjust based on VRAM
    generator=torch.Generator(device).manual_seed(42),
).frames[0]

export_to_video(frames, "output.mp4", fps=8)
print(f"Video saved! Camera motion: {camera_action}, Total frames: {len(frames)}")


The config attributes {'motion_activation_fn': 'geglu', 'motion_attention_bias': False, 'motion_cross_attention_dim': None} were passed to MotionAdapter, but are not expected and will be ignored. Please verify your config.json configuration file.
Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\Admin\.cache\huggingface\hub\models--frankjoshua--toonyou_beta6\snapshots\9bb5e1c5be60ab664ec80bff429126e0844ac9fe\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\Admin\.cache\huggingface\hub\models--frankjoshua--toonyou_beta6\snapshots\9bb5e1c5be60ab664ec80bff429126e0844ac9fe\unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  17%|█▋        | 1/6 [00:01<00:05,  1.09s/it]d:\projects\stable-ads\.venv\Lib\site-packages\transformers\models\clip\feature_extraction_clip.py:30: FutureWarning: The class CLIPFeatureExtr

Video saved! Camera motion: zoom_in, Total frames: 16
